# Ensemble (Voting Classifier)

## Foco: Combinar os melhores modelos em um "Comitê".

**Justificativa Estratégica:** Nenhum modelo individual é perfeito. Cada um dos nossos quatro campeões (Random Forest, XGBoost, CatBoost, MLPClassifier) possui um "padrão de pensamento" e um conjunto de forças e fraquezas ligeiramente diferentes. A teoria do "Wisdom of the Crowd" (Sabedoria da Multidão) sugere que a decisão coletiva de um grupo é frequentemente superior à decisão de qualquer modelo individual.

**Hipótese:** Ao combinar as previsões de probabilidade dos nossos melhores modelos (uma técnica chamada **Soft Voting**), podemos criar um modelo superior (ensemble) que é mais robusto, generaliza melhor e alcança uma performance geral (F1-score) superior à de qualquer um de seus componentes. Este será o nosso classificador supervisionado final.

**Metodologia:**
1. Carregar os quatro modelos campeões.
2. Gerar as previsões de probabilidade de cada modelo para o conjunto de teste.
3. Calcular a média dessas probabilidades.
4. A classe final prevista será aquela com a maior probabilidade média.
5. Avaliar a performance do ensemble e compará-la com a dos modelos individuais.

### 1. Importação das Bibliotecas:

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [14, 10]
plt.rcParams['font.size'] = 12

### 2. Carregamento Centralizado de Dados e Modelos

 Esta etapa é idêntica à do notebook de comparação.
 Vamos carregar todos os artefatos e preparar os conjuntos de teste (realista e padronizado) com a metodologia correta para evitar qualquer vazamento de dados.

In [2]:
# --- 1. Define os caminhos ---
try:
    project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
except:
    project_root = os.getcwd()

model_dir = os.path.join(project_root, 'models')
data_dir = os.path.join(project_root, 'MachineLearningCSV', 'MachineLearningCVE')

print(f"Diretório de modelos: {model_dir}")
print(f"Diretório de dados: {data_dir}")

# --- 2. Carrega os artefatos de pré-processamento ---
print("\nCarregando artefatos de pré-processamento...")
try:
    label_encoder = joblib.load(os.path.join(model_dir, 'label_encoder.joblib'))
    features_realistas = joblib.load(os.path.join(model_dir, 'features_realistas.joblib'))
    print("  - Artefatos carregados com sucesso.")
except Exception as e:
    raise RuntimeError(f"ERRO: Não foi possível carregar os artefatos. Verifique o notebook 01. Erro: {e}")

# --- 3. Carrega e limpa o dataset completo ---
print("\nCarregando e limpando o dataset completo...")
todos_os_arquivos = glob.glob(os.path.join(data_dir, "*.csv"))
dados = pd.concat((pd.read_csv(f, low_memory=False) for f in todos_os_arquivos), ignore_index=True)
dados.columns = dados.columns.str.strip()
for col in dados.columns:
    if dados[col].dtype == 'object' and col != 'Label':
        dados[col] = pd.to_numeric(dados[col], errors='coerce')
dados.replace([np.inf, -np.inf], np.nan, inplace=True)
dados.dropna(inplace=True)
print("  - Dataset carregado e limpo.")

# --- 4. Separa features e labels ---
X_bruto = dados.drop('Label', axis=1)
y_bruto = dados['Label']
y_codificado = label_encoder.transform(y_bruto)
X_realista = X_bruto[features_realistas]

# --- 5. Metodologia de Preparação dos Dados de Teste ---
print("\nPreparando conjuntos de teste (com prevenção de data leakage)...")

# Dividir os dados brutos em treino e teste
X_treino_realista, X_teste_realista, y_treino, y_teste = train_test_split(
    X_realista, y_codificado, test_size=0.3, random_state=42, stratify=y_codificado
)
print("  - Dados divididos em treino e teste.")

# Treinar o Scaler APENAS com os dados de treino
scaler = StandardScaler().fit(X_treino_realista)
print("  - StandardScaler treinado apenas com dados de treino.")

# Transformar o conjunto de teste para criar a versão padronizada
X_teste_padronizado = scaler.transform(X_teste_realista)
print("  - Conjunto de teste padronizado criado sem vazamento de dados.")

print("\nConjuntos de teste prontos:")
print(f"  - Shape do X_teste_realista (para modelos de árvore): {X_teste_realista.shape}")
print(f"  - Shape do X_teste_padronizado (para MLP): {X_teste_padronizado.shape}")
print(f"  - Shape do y_teste (nosso alvo): {y_teste.shape}")

# --- 6. Carrega os modelos ---
print("\nCarregando modelos salvos...")

modelos = {}
nomes_modelos = [
    'random_forest_model',
    'xgboost_model',
    'catboost_model',
    'mlp_model'
]

for nome in nomes_modelos:
    caminho_modelo = os.path.join(model_dir, f"{nome}.joblib")
    try:
        modelos[nome] = joblib.load(caminho_modelo)
        print(f"  - Modelo '{nome}' carregado.")
    except FileNotFoundError:
        print(f"  - AVISO: Modelo '{nome}' não encontrado em '{caminho_modelo}'.")

print("\n--- Setup concluído ---")

Diretório de modelos: /home/henrique/PycharmProjects/ndr-tcc/models
Diretório de dados: /home/henrique/PycharmProjects/ndr-tcc/MachineLearningCSV/MachineLearningCVE

Carregando artefatos de pré-processamento...
  - Artefatos carregados com sucesso.

Carregando e limpando o dataset completo...
  - Dataset carregado e limpo.

Preparando conjuntos de teste (com prevenção de data leakage)...
  - Dados divididos em treino e teste.
  - StandardScaler treinado apenas com dados de treino.
  - Conjunto de teste padronizado criado sem vazamento de dados.

Conjuntos de teste prontos:
  - Shape do X_teste_realista (para modelos de árvore): (848363, 14)
  - Shape do X_teste_padronizado (para MLP): (848363, 14)
  - Shape do y_teste (nosso alvo): (848363,)

Carregando modelos salvos...
  - Modelo 'random_forest_model' carregado.
  - Modelo 'xgboost_model' carregado.
  - Modelo 'catboost_model' carregado.
  - Modelo 'mlp_model' carregado.

--- Setup concluído ---


### 3. Implementação do Ensemble (Soft Voting)

 Chegou a hora de combinar a "opinião" dos nossos especialistas.

 O processo de Soft Voting manual será:
 1. Para cada modelo, obter o array de previsões de probabilidade. Cada array terá o shape `(n_amostras, n_classes)`.
 2. Somar todos esses arrays. O resultado será um array final onde cada célula contém a "confiança" combinada para aquela amostra e classe.
 3. Para cada amostra, encontrar o índice da classe que recebeu a maior soma de probabilidades.
 4. Esse índice será a previsão final do nosso ensemble.

In [3]:
print("--- Iniciando o processo de Soft Voting ---")

# Lista para armazenar os arrays de probabilidade de cada modelo
lista_de_probas = []

# Gerar as probabilidades para cada modelo
for nome, modelo in modelos.items():
    print(f"Gerando probabilidades para: {nome}...")

    # Seleciona o conjunto de teste correto
    if nome == 'mlp_model':
        X_teste = X_teste_padronizado
    else:
        X_teste = X_teste_realista

    # Gera e armazena as probabilidades
    probas = modelo.predict_proba(X_teste)
    lista_de_probas.append(probas)
    print(f"  -> Shape das probabilidades: {probas.shape}")

# Somar as probabilidades
# np.sum() com axis=0 soma os arrays elemento por elemento.
soma_das_probas = np.sum(lista_de_probas, axis=0)
print(f"\nShape do array com a soma das probabilidades: {soma_das_probas.shape}")

# Encontrar a classe com a maior probabilidade somada
# np.argmax() com axis=1 encontra o índice do valor máximo em cada linha (amostra).
y_pred_ensemble = np.argmax(soma_das_probas, axis=1)
print(f"Shape do array de previsões finais do ensemble: {y_pred_ensemble.shape}")

print("\n--- Processo de Soft Voting concluído ---")

--- Iniciando o processo de Soft Voting ---
Gerando probabilidades para: random_forest_model...
  -> Shape das probabilidades: (848363, 15)
Gerando probabilidades para: xgboost_model...
  -> Shape das probabilidades: (848363, 15)
Gerando probabilidades para: catboost_model...
  -> Shape das probabilidades: (848363, 15)
Gerando probabilidades para: mlp_model...
  -> Shape das probabilidades: (848363, 15)

Shape do array com a soma das probabilidades: (848363, 15)
Shape do array de previsões finais do ensemble: (848363,)

--- Processo de Soft Voting concluído ---


### 4. Avaliação Final do Ensemble

 Agora, vamos comparar a performance do nosso ensemble com os modelos individuais.

 Mesmo que o aumento seja pequeno, um modelo ensemble é geralmente preferível por ser mais robusto e menos propenso a erros específicos de um único algoritmo.

In [4]:
print("--- Relatório de Classificação (Ensemble - Soft Voting) ---")

# Gera o relatório de classificação para as previsões do ensemble
print(classification_report(y_teste, y_pred_ensemble, target_names=label_encoder.classes_.astype(str), zero_division=0))

# Calcula as métricas gerais
acc_ensemble = accuracy_score(y_teste, y_pred_ensemble)
f1_ensemble = f1_score(y_teste, y_pred_ensemble, average='weighted')

print("\n--- Comparação Final de Performance ---")

# --- Recria a tabela de performance para comparação ---
print("Recriando tabela de performance para comparação...")
lista_de_resultados_individuais = []
for nome, modelo in modelos.items():
    if nome == 'mlp_model':
        X_t = X_teste_padronizado
    else:
        X_t = X_teste_realista
    y_p = modelo.predict(X_t)
    lista_de_resultados_individuais.append({
        'Modelo': nome.replace('_model', '').replace('_', ' ').title(),
        'Acurácia': accuracy_score(y_teste, y_p),
        'F1-Score Ponderado': f1_score(y_teste, y_p, average='weighted')
    })
df_comparacao = pd.DataFrame(lista_de_resultados_individuais)

# Adiciona o resultado do ensemble à tabela
ensemble_resultado = pd.DataFrame([{
    'Modelo': 'Ensemble (Soft Voting)',
    'Acurácia': acc_ensemble,
    'F1-Score Ponderado': f1_ensemble
}])
df_final = pd.concat([df_comparacao, ensemble_resultado], ignore_index=True)

# Ordena e mostra a tabela final
df_final = df_final.sort_values(by='F1-Score Ponderado', ascending=False).reset_index(drop=True)

display(df_final.style.format({
    'Acurácia': '{:.4f}',
    'F1-Score Ponderado': '{:.5f}' # 5 casas decimais para ver a diferença
}).background_gradient(cmap='viridis', subset=['F1-Score Ponderado']))

--- Relatório de Classificação (Ensemble - Soft Voting) ---
                            precision    recall  f1-score   support

                    BENIGN       1.00      0.98      0.99    681396
                       Bot       0.99      0.71      0.83       587
                      DDoS       0.93      1.00      0.96     38408
             DoS GoldenEye       0.97      0.28      0.44      3088
                  DoS Hulk       0.95      1.00      0.97     69037
          DoS Slowhttptest       0.67      0.74      0.70      1650
             DoS slowloris       0.90      0.56      0.69      1739
               FTP-Patator       0.88      1.00      0.94      2380
                Heartbleed       0.00      0.00      0.00         3
              Infiltration       0.00      0.00      0.00        11
                  PortScan       0.83      1.00      0.91     47641
               SSH-Patator       1.00      1.00      1.00      1769
  Web Attack � Brute Force       0.75      0.07      0.

,Modelo,Acurácia,F1-Score Ponderado
0,Random Forest,0.9786,0.97810
1,Catboost,0.9784,0.97784
2,Ensemble (Soft Voting),0.9784,0.97783
3,Xgboost,0.9774,0.97687
4,Mlp,0.9413,0.94086


### 5. Encapsulando e Salvando o Ensemble

 Para que nosso ensemble possa ser usado em outros lugares (como em uma aplicação web), não podemos apenas salvar a lógica. Precisamos encapsulá-la em um objeto, assim como os outros modelos.

 Vamos criar uma classe Python customizada que:
 1. Armazena nossos quatro modelos campeões.
 2. Possui um método `.predict()` que implementa a lógica de Soft Voting.

 Este objeto, contendo os outros quatro, será o nosso artefato final.

In [7]:
from sklearn.base import BaseEstimator, ClassifierMixin

class SoftVotingEnsemble(BaseEstimator, ClassifierMixin):
    """
    Uma classe customizada para nosso ensemble de Soft Voting.
    Ela armazena os modelos treinados e implementa a lógica de previsão.
    """
    def __init__(self, models_dict):
        self.models_dict = models_dict
        # O scaler é essencial para o MLP, então o armazenamos também.
        self.scaler = StandardScaler()

    def fit(self, X_train_realista, y_train):
        """
        Treina o scaler. Os modelos já estão treinados.
        """
        print("Ajustando o Scaler com os dados de treino...")
        self.scaler.fit(X_train_realista)
        print("Scaler ajustado.")
        return self

    def predict(self, X_realista):
        """
        Gera a previsão final usando a lógica de Soft Voting.
        """
        # Prepara a versão padronizada dos dados para o MLP
        X_padronizado = self.scaler.transform(X_realista)

        lista_de_probas = []
        for nome, modelo in self.models_dict.items():
            if nome == 'mlp_model':
                X_teste = X_padronizado
            else:
                X_teste = X_realista

            probas = modelo.predict_proba(X_teste)
            lista_de_probas.append(probas)

        soma_das_probas = np.sum(lista_de_probas, axis=0)
        y_pred_final = np.argmax(soma_das_probas, axis=1)

        return y_pred_final

# --- Criação e Teste do Objeto Ensemble ---

print("Criando o objeto SoftVotingEnsemble...")
# Passa o dicionário com nossos modelos já carregados
ensemble_model = SoftVotingEnsemble(models_dict=modelos)

# "Treina" o ensemble (na verdade, apenas treina o scaler interno)
# Usamos o X_treino_realista que foi criado na Célula 5
ensemble_model.fit(X_treino_realista, y_treino)

# Testa o métod .predict() para garantir que funciona
print("\nTestando o método .predict() do ensemble...")
y_pred_obj_teste = ensemble_model.predict(X_teste_realista)

# Compara com o resultado anterior para garantir que são idênticos
np.testing.assert_array_equal(y_pred_obj_teste, y_pred_ensemble)
print("SUCESSO: A previsão do objeto é idêntica à previsão manual. O objeto está funcionando corretamente.")

Criando o objeto SoftVotingEnsemble...
Ajustando o Scaler com os dados de treino...
Scaler ajustado.

Testando o método .predict() do ensemble...
SUCESSO: A previsão do objeto é idêntica à previsão manual. O objeto está funcionando corretamente.


In [8]:
# --- Salva o objeto Ensemble final ---
# Este único arquivo conterá nossa classe e os 4 modelos campeões.
ensemble_path = os.path.join(model_dir, 'ensemble_model.joblib')
joblib.dump(ensemble_model, ensemble_path)

print(f"Modelo Ensemble final salvo em: {ensemble_path}")

# --- Verificação Final ---
print("\n--- Verificando arquivos salvos no diretório 'models' ---")
print(sorted(os.listdir(model_dir)))

Modelo Ensemble final salvo em: /home/henrique/PycharmProjects/ndr-tcc/models/ensemble_model.joblib

--- Verificando arquivos salvos no diretório 'models' ---
['catboost_model.joblib', 'ensemble_model.joblib', 'features_realistas.joblib', 'isolation_forest_model.joblib', 'label_encoder.joblib', 'mlp_model.joblib', 'random_forest_model.joblib', 'top_15_features.joblib', 'xgboost_model.joblib']
